In [22]:
import pandas as pd


In [23]:
df = pd.read_csv("../../data/processed/Online_Retail_flagged.csv", encoding="ISO-8859-1", parse_dates=["InvoiceDate"])
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,is_cancelled,is_return,is_missing_customer,is_invalid_price,is_outlier_quantity,Revenue,YearMonth,Hour,Weekday
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,False,False,False,False,False,15.30,2010-12,8,Wednesday
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,False,False,False,False,20.34,2010-12,8,Wednesday
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,False,False,False,False,False,22.00,2010-12,8,Wednesday
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,False,False,False,False,20.34,2010-12,8,Wednesday
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,False,False,False,False,20.34,2010-12,8,Wednesday


In [24]:
return_late = df["is_return"].mean()
print(f'Return Rate: {return_late:.2%}')

Return Rate: 1.97%


In [25]:
cancelled_rate = df["is_cancelled"].mean()
print(f'Cancelled Rate: {cancelled_rate:.2%}')

Cancelled Rate: 1.72%


In [26]:
revenue_lost = df[df["is_return"]]["Revenue"].abs().sum()
print(f'Total Revenue Lost Due to Returns: ${revenue_lost:,.2f}')

Total Revenue Lost Due to Returns: $893,979.73


In [27]:
return_rate_product = (
    df.groupby("Description")["is_return"].mean()
    .sort_values(ascending=False)
    .head(10)
)
print(return_rate_product)

Description
wrongly coded-23343             1.0
water damage                    1.0
wrongly sold sets               1.0
wrongly sold as sets            1.0
wrongly marked. 23343 in box    1.0
water damaged                   1.0
thrown away-can't sell.         1.0
thrown away-can't sell          1.0
wet rusty                       1.0
thrown away                     1.0
Name: is_return, dtype: float64


In [28]:
kpi_flagged = pd.DataFrame({
    "metric": [
        "Return Rate",
        "Cancel Rate",
        "Revenue Lost",
        "Return Impact"
    ],
    "value": [
        df["is_return"].mean(),
        df["is_cancelled"].mean(),
        df[df["is_return"]]["Revenue"].abs().sum(),
        df[df["is_return"]]["Revenue"].abs().sum() / df["Revenue"].sum()
    ]
})
kpi_flagged.to_csv("../../data/kpi/kpi_flagged_summary.csv", index=False)

In [29]:
net_revenue = df["Revenue"].sum()
print(f"Net Revenue: ${net_revenue:,.2f}")

Net Revenue: $9,726,006.95


In [30]:
gross_revenue = df[df["Quantity"] > 0]["Revenue"].sum()
lost_revenue = df[df["Quantity"] < 0]["Revenue"].abs().sum()

print(gross_revenue, lost_revenue)

10619986.68 893979.73


In [31]:
return_analysis = df[df["is_return"]]

return_analysis_stats = pd.DataFrame({
    "metric": ["Total Returns", "Avg Return Value"],
    "value": [
        return_analysis.shape[0],
        return_analysis["Revenue"].abs().mean()
    ]
})

return_analysis_stats.to_csv("../../data/kpi/return_analysis.csv", index=False)

In [32]:
lost_by_product = (
    df[df["is_return"]]
    .groupby("Description")["Revenue"]
    .apply(lambda x: x.abs().sum())
    .sort_values(ascending=False)
    .reset_index()
)

lost_by_product.to_csv("../../data/kpi/lost_by_product.csv", index=False)

In [33]:
df["Month"] = df["InvoiceDate"].dt.to_period("M")

return_by_time = (
    df[df["is_return"]]
    .groupby("Month")["Revenue"]
    .apply(lambda x: x.abs().sum())
    .reset_index()
)

return_by_time.to_csv("../../data/kpi/return_by_time.csv", index=False)

In [34]:
cancel_analysis = df[df["is_cancelled"]]

cancel_stats = pd.DataFrame({
    "metric": ["Total Cancelled Orders"],
    "value": [cancel_analysis.shape[0]]
})

cancel_stats.to_csv("../../data/kpi/cancel_analysis.csv", index=False)